## How To Test Models

In this short notebook we will learn how to calculate metrics for our models.

### Metrics for 3D annotations

Since not all users of darts devkit will use our evaluation tools we made decision to keep it as a optional dependency. Run below cell to install available evaluation tool.

In [ ]:
pip install darts_devkit[polygon_overlap_evaluator]

Below functions will help as create json with annotations that we will then run our ap metrics on. The DARTSAnnotation file format is presented below:

class Box(BaseModel):
    """Data type representing a bounding box in DARTS format."""

    center: Annotated[list[float], Len(min_length=3, max_length=3)]
    """Center of a box [x, y, z]."""
    size: Annotated[list[PositiveFloat], Len(min_length=3, max_length=3)]
    """Size of a box [x, y, z]."""
    orientation: Annotated[list[Annotated[float, Field(ge=-1, le=1)]], Len(min_length=4, max_length=4)]
    """Quaternion rotation [w, x, y, z]."""
    name: str
    """Class name."""
    score: Annotated[float, Field(ge=0.0, le=1.0)]
    """Detector confidence score."""
    model_config = ConfigDict(extra="forbid")


class Frame(BaseModel):
    """Data type representing a frame in DARTS format."""

    sample_token: str
    boxes: list[Box]


class DARTSAnnotations(BaseModel):
    """Data type representing annotations in DARTS format."""

    sequences: dict[str, list[Frame]]

In [ ]:
def create_box_json():
    return {"center": [1, 1, 1],
           "size": [2, 2, 2],
           "orientation": [1, 0, 0, 0],
           "name":"multi_track_vehicle.car",
           "score": 0.4}
def create_annotation_json(darts):
    annotations = {"sequences": {}}
    for scene in darts.scene.all():
        frames = []
        for sample in darts.get_samples_from_scene(scene.token):
            frames.append({"sample_token": sample.token,
                          "boxes": [create_box_json()]})
        annotations["sequences"][scene.token] = frames
    return annotations

What we also need is instance of PolygonOverlaEvaluationConfig object with our flexible config and we can run our evaluation.

class ClassThresholdConfig(BaseModel):
    """IoU threshold for a single class.

    If IOU between detection and ground truth in this class is less than iou_threshold
    then such pairing is not taken into account during global IOU maximalization.
    """

    class_name: str
    iou_threshold: Annotated[float, Field(ge=0.0, le=1.0)]


class PolygonOverlaoEvaluationConfig(BaseModel):
    """User configuration for evaluation."""

    class_thresholds: list[ClassThresholdConfig]
    """List of ClassThresholdConfig. Only ap of classes given in this list are computed."""
    num_score_thresholds: Annotated[int, Field(ge=1)]
    """Number of evenly spaced score thresholds for detections."""
    pr_curve_density: Annotated[float, Field(ge=0.0, le=1.0)]
    """How dense pr curve should be or recall axis."""
    pr_rounding: Annotated[int, Field(ge=0)]
    """Rounding point of various results during calculation pr_curve and ap."""
    min_gt_lidar_points: Annotated[int, Field(ge=0)]
    """Minumum number of lidar points inside ground truth box for it to be taken into account."""

To make it easier for later development on other evaluation software we use Registry Pattern. For now we implemented 3D annotation evaluation that uses IOU based on polygon overlap. Below code registers available RerunVisualizer and opens rerun window with first scene in dataset.

In [ ]:
import darts_devkit.evaluation as ev
from darts_devkit import DARTS, EvaluateRegistry
from darts_devkit.evaluation.evaluation_models import DARTSAnnotations
from darts_devkit.evaluation.polygon_overlap_evaluator import PolygonOverlapEvaluationConfig, ClassThresholdConfig

ev.register_polygon_overlap_evaluator()
evaluator = EvaluateRegistry().get("PolygonOverlapEvaluator")()
darts=DARTS("/data/darts", "test")
config = PolygonOverlapEvaluationConfig(class_thresholds=[ClassThresholdConfig(class_name="multi_track_vehicle.car",  iou_threshold=0.5)], 
                                                         num_score_thresholds=10, pr_curve_density=0.05, pr_rounding=6, min_gt_lidar_points=0)
annotations = DARTSAnnotations(**create_annotation_json(darts))

results = evaluator.evaluate(darts, annotations, config)
print(results)